In [ ]:
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt


from pathlib import Path
import numpy as np


def load_specmaps(folder, ppm_range=None, ppm_min=None, ppm_max=None):
    """
    ppm_range: tuple (start_ppm, end_ppm) z. B. (1, 6)
    ppm_min/max: original ppm range der Daten
    """

    folder = Path(folder)
    keys = ["LCMInput", "LCMFit", "LCMBaseline", "water", "Glc", "Glx", "Lac"]
    data = {k: np.load(folder / f"{k}_spectra.npy") for k in keys}

    # --- optional cropping ---
    if ppm_range is not None:
        N = data["LCMInput"].shape[3]

        # original ppm Achse rekonstruieren
        ppm_full = np.linspace(ppm_max, ppm_min, N)

        # gewünschte range
        mask = (ppm_full <= ppm_range[1]) & (ppm_full >= ppm_range[0])

        # crop alle arrays
        for k in keys:
            data[k] = data[k][:, :, :, mask, :]

        data["ppm"] = ppm_full[mask]

    return data


def plot_voxel_fit(data, voxel, rep, ppm=None, title=""):
    x, y, z = voxel

    inp = data["LCMInput"][x, y, z, :, rep]
    fit = data["LCMFit"][x, y, z, :, rep]
    baseline = data["LCMBaseline"][x, y, z, :, rep]

    water = data["water"][x, y, z, :, rep]
    glc = data["Glc"][x, y, z, :, rep]
    glx = data["Glx"][x, y, z, :, rep]
    lac = data["Lac"][x, y, z, :, rep]

    residual = inp - (fit)

    if ppm is None:
        ppm = np.arange(inp.shape[0])

    fig, axs = plt.subplots(1, 2, figsize=(11, 4), sharex=True)

    axs[0].plot(ppm, inp, label="LCM input")
    axs[0].plot(ppm, fit, label="LCM fit")
    axs[0].plot(ppm, residual, label="Residual")
    axs[0].set_title(f"{title} input + fit")
    axs[0].legend()

    axs[1].plot(ppm, fit, label="Total fit")
    axs[1].plot(ppm, baseline, label="Baseline")
    axs[1].plot(ppm, water, label="HDO")
    axs[1].plot(ppm, glc, label="Glc")
    axs[1].plot(ppm, glx, label="Glx")
    axs[1].plot(ppm, lac, label="Lac")
    axs[1].set_title(f"{title} fit decomposition")
    axs[1].legend()

    for ax in axs:
        ax.set_xlabel("Chemical shift / spectral index")
        ax.grid(alpha=0.2)

    plt.tight_layout()
    plt.show()

In [ ]:
healthy = load_specmaps("SpecMaps/REFIT_P08_deep_tMPPCA_5D")
tumor = load_specmaps("SpecMaps/REFITTED_Tumor_1_Part_2_Proposed")

Sim =load_specmaps("SpecMaps/Lesion_double_deep_tmppca_6")

In [ ]:
# Sim 

x,y,z =14, 11, 15 # 14, 11, 12 good for tumor x=13, y=15, z=11,

plot_voxel_fit(Sim, voxel=(x, y, z), rep=-2, title="Sim")

In [ ]:
# Healthy

x,y,z =8, 11, 11 # 14, 11, 12 good for tumor x=13, y=15, z=11,

plot_voxel_fit(healthy, voxel=(x, y, z), rep=-1, title="Healthy")

In [ ]:
x,y,z =13, 15, 11 # 14, 11, 12 good for tumor x=13, y=15, z=11,

plot_voxel_fit(tumor, voxel=(x, y, z), rep=-2, title="Tumor")



In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_lcmodel_figure(datasets, voxels, reps, titles, save_path=None):

    fig, axs = plt.subplots(1, len(datasets), figsize=(14, 4), sharex=True)

    for i, (data, voxel, rep, title) in enumerate(zip(datasets, voxels, reps, titles)):
        ax = axs[i]
        x, y, z = voxel

        inp = data["LCMInput"][x, y, z, :, rep]
        fit = data["LCMFit"][x, y, z, :, rep]

        water = data["water"][x, y, z, :, rep]
        glc = data["Glc"][x, y, z, :, rep]
        glx = data["Glx"][x, y, z, :, rep]
        lac = data["Lac"][x, y, z, :, rep]

        residual = inp - fit
        x_axis = np.arange(inp.shape[0])

        # --- main spectrum ---
        ax.plot(x_axis, inp, color="black", lw=1.5, label="Proposed")
        ax.plot(x_axis, fit, color="red", lw=1.2, linestyle="--", label="LCModel fit")

        # --- offsets ---
        scale = np.max(np.abs(inp))
        spacing = 0.35 * scale
        offset = -spacing

        res_off = offset
        ax.plot(x_axis, residual + res_off, color="0.5", lw=0.7)
        offset -= spacing

        water_off = offset
        ax.plot(x_axis, water + water_off, lw=1)
        offset -= spacing

        glc_off = offset
        ax.plot(x_axis, glc + glc_off, lw=1)
        offset -= spacing

        glx_off = offset
        ax.plot(x_axis, glx + glx_off, lw=1)
        offset -= spacing

        lac_off = offset
        ax.plot(x_axis, lac + lac_off, lw=1)

        # --- labels links ---
        x_pos = x_axis[0] + 2

        ax.text(x_pos, residual[0] + res_off, "Residual", fontsize=9, va="bottom")
        ax.text(x_pos, water[0] + water_off, "HDO", fontsize=9, va="bottom")
        ax.text(x_pos, glc[0] + glc_off, "Glc", fontsize=9, va="bottom")
        ax.text(x_pos, glx[0] + glx_off, "Glx", fontsize=9, va="bottom")
        ax.text(x_pos, lac[0] + lac_off, "Lac", fontsize=9, va="bottom")

        # --- ppm Beschriftung (6 → 1) ---
        N = len(x_axis)
        ppm_ticks = [6, 5, 4, 3, 2, 1]
        tick_positions = np.linspace(0, N - 1, len(ppm_ticks))

        ax.set_xticks(tick_positions)
        ax.set_xticklabels(ppm_ticks)
        ax.set_xlabel("Chemical shift [ppm]")

        # --- cosmetics ---
        ax.legend(loc="upper right", fontsize=8, frameon=False)
        ax.set_title(title)
        ax.set_yticks([])              # y-Ticks entfernen
        ax.grid(alpha=0.15)            # dezentes Grid

    plt.tight_layout()

    # --- optional speichern ---
    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()

In [ ]:
sim_data = load_specmaps("SpecMaps/Lesion_double_deep_tmppca_6", ppm_range=(1, 6), ppm_min=0.5,
    ppm_max=8)

healthy_data = load_specmaps("SpecMaps/REFIT_P08_deep_tMPPCA_5D", ppm_range=(1, 6), ppm_min=1,
    ppm_max=8)

tumor_data = load_specmaps("SpecMaps/REFITTED_Tumor_1_Part_2_Proposed", ppm_range=(1, 6), ppm_min=1,
    ppm_max=8)

sim_voxel = (14, 11, 15)
healthy_voxel = (8, 11, 11)
tumor_voxel = (13, 15, 11)

rep = -1  # z. B. später Zeitpunkt

plot_lcmodel_figure(
    datasets=[sim_data, healthy_data, tumor_data],
    voxels=[sim_voxel, healthy_voxel, tumor_voxel],
    reps=[rep, rep, rep],
    titles=["Simulation", "Healthy", "Tumor"],
    save_path="SavedGraphics/FittingFigure.pdf"
)